In [ ]:
# Removendo mensagens de erro:
import warnings

warnings.filterwarnings(
    "ignore",
    message=r".*is_datetime64tz_dtype is deprecated.*",
    category=DeprecationWarning,
)
# Limpando outputs!
from IPython.display import clear_output

## 1. Extraindo os Dados:

#### Importando bibliotecas essenciais para a Análise de Dados:

In [ ]:
# Importando bibliotecas:
from __future__ import annotations                          # Adia a avaliação das type hints (permite "forward references", ajuda a evitar imports circulares e melhora performance em alguns casos)...
from pathlib import Path                                    # Pegando o endereço do arquivo...
from glob import glob                                       # Busca de arquivos por padrões...
import pandas as pd                                         # Análise e manipulação de dados...
import numpy as np                                          # Tratamento de Dados...
import warnings                                             # Controle de avisos...
import sqlite3                                              # Biblioteca para criar a base de dados em Data Warehouse...
import os                                                   # Operações com arquivos e sistema...

pd.set_option('mode.chained_assignment', None)              # Desativa avisos de atribuição encadeada...
warnings.filterwarnings("ignore", category=FutureWarning)   # Oculta avisos de futura depreciação...


#### Consumindo os Dados dos arquivos Parquet:

In [ ]:
# Caminhos das pastas com os arquivos '.parquet':
pasta_ativos = R"..\Base de Dados Original\parquet\Ativos\Ativos.parquet"
pasta_desligados = R"..\Base de Dados Original\parquet\Desligados\Desligados.parquet"
pasta_mensalistas = R"..\Base de Dados Original\parquet\Mensalistas"
pasta_gptw = R"..\Base de Dados Original\parquet\GPTW\GPTW.parquet"
pasta_executivos = R"..\Base de Dados Original\parquet\Executivos"
pasta_horistas = R"..\Base de Dados Original\parquet\Horistas"
pasta_movimentacoes = R"..\Base de Dados Original\parquet\Movimentacoes\Movimentacoes.parquet"

# Lista todos os arquivos '.parquet' em cada pasta:
arquivos_parquet_executivos = glob(os.path.join(pasta_executivos, "*.parquet"))
arquivos_parquet_mensalistas = glob(os.path.join(pasta_mensalistas, "*.parquet"))
arquivos_parquet_horistas = glob(os.path.join(pasta_horistas, "*.parquet"))

# Carrega os arquivos e adiciona a coluna 'Cargo':
tables = [
    pd.read_parquet(caminho).assign(Cargo=cargo)
    for arquivos, cargo in zip([arquivos_parquet_executivos, arquivos_parquet_mensalistas, arquivos_parquet_horistas], ['E', 'M', 'H'])
    for caminho in arquivos
]

# Carrega o arquivo para dentro do Data Frame:
df_desligados = pd.read_parquet(pasta_desligados)
df_ativos = pd.read_parquet(pasta_ativos)
df_gptw = pd.read_parquet(pasta_gptw)
df_movimentacoes = pd.read_parquet(pasta_movimentacoes)

In [ ]:
# === Resumo final ===
total_linhas = sum(len(df) for df in tables)
print("\n📊 RESUMO GERAL")
print(f"Arquivos executivos: {len(arquivos_parquet_executivos)}")
print(f"Arquivos mensalistas: {len(arquivos_parquet_mensalistas)}")
print(f"Arquivos horistas: {len(arquivos_parquet_horistas)}")
print(f"Total de DataFrames lidos: {len(tables)}")
print(f"Total de linhas combinadas: {total_linhas:,}")

## 2. Tratando os Dados:

#### Definição das Funções para Transformação e Tratamento de Dados:
1. **Renomear Colunas:**  
   Função que renomeia as colunas do DataFrame com base em um dicionário de mapeamento, garantindo padronização e clareza nos nomes.
2. **Mover Colunas:**  
   Função que move as colunas do DataFrame com base em um dicionário de mapeamento, garantindo padronização e clareza nos nomes.

In [ ]:
# Função para renomear coluna:
def renomear_coluna(df, col, newname):
    df = df.rename(columns={col: newname})
    return df

# Função para mover coluna
def mover_coluna(df, col, nova_posicao):
    colunas = list(df.columns)
    colunas.insert(nova_posicao, colunas.pop(colunas.index(col)))
    return df[colunas]

#### Aplicando Funções a lista de tabelas:

In [ ]:
# Aplicando a função a todas as colunas de todas as tabelas:
for i in range(len(tables)):
    clear_output(wait=True)
    print(f"Processando tabela {i+1}/{len(tables)}", end="", flush=True)
    tables[i] = renomear_coluna(tables[i], tables[i].columns[0], 'Cadastro')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[1], 'Coletor')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[2], 'Descricao Coletor')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[3], 'Nome')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[4], 'Filial')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[5], 'Data')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[6], 'Hora')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[7], 'Acesso')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[8], 'Dir')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[9], 'Tipo')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[10], 'Uso Acesso')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[11], 'Planta')
#'Cadastro', 'Nome', 'Filial', 'Data', 'Hora', 'Dir', 'Tipo', 'Acesso', 'Coletor', 'Planta', 'Descricao Coletor', 'Uso Acesso', 'Cargo'...

# Aplicando a função de renomear o nome da coluna nas colunas da tabela 'df_ativos':
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[0], 'Cadastro')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[1], 'Cargo')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[2], 'Nome completo')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[3], 'Heranca')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[4], 'Cargo Especifico')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[5], 'Expatriado')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[6], 'Departamento')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[7], 'Área')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[8], 'Local')

# Aplicando a função de renomear o nome da coluna 'Departamento' da tabela 'df_desligados':
df_desligados = mover_coluna(df_desligados, 'Diretoria RH', 2)
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[0], 'Cadastro')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[2], 'Heranca')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[3], 'Cargo Especifico')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[4], 'Departamento')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[5], 'Área')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[6], 'Local')

# Aplicando a função de renomear o nome da coluna nas colunas da tabela 'df_gptw':
df_gptw = renomear_coluna(df_gptw, df_gptw.columns[0], 'Área')

# Aplicando a função de renomear o nome da coluna nas colunas da tabela 'df_movimentacoes':
df_movimentacoes = renomear_coluna(df_movimentacoes, df_movimentacoes.columns[0], 'Cadastro')

#### Concatenar todas as tabelas da lista em um único Data Frame:

In [ ]:
# Concatenando todas as tabelas em um único DataFrame:
df_ronda = pd.concat(tables, ignore_index=True)

# Ordenando as colunas na ordem desejada:
df_ronda = df_ronda[['Cadastro', 'Nome', 'Filial', 'Data', 'Hora', 'Dir', 'Tipo', 'Acesso', 'Planta', 'Coletor', 'Descricao Coletor', 'Uso Acesso', 'Cargo']]
df_movimentacoes = df_movimentacoes[
    ['Cadastro', 'Abreviação UO antiga', 'Desc. UO antiga', 'Desc. Cargo antigo',
    'Abreviação UO', 'Desc. UO', 'Desc. Cargo', 'Tipo da Movimentação', 'Data de alteração']
]
df_ronda = df_ronda.drop(df_ronda.index[-1]) # Remover última linha de total...

#### Diferenciando as pessoas que foram demitidas ou que não serão analisadas e inserindo todos os df_ativos delas:

In [ ]:
# Criando toda a função para encontrar as pessoas desligadas não registradas nas bases 'df_ativos' e 'df_desligados':
TARGET_COLS = [
    "Cadastro", "Cargo",
    "Nome completo", "Heranca", "Cargo Especifico",
    "Departamento", "Área", "Local", "Data_Adm", "Data_Dem"
]

def _to_dt(s, dayfirst=True):
    return pd.to_datetime(s, errors="coerce", dayfirst=dayfirst)

def _coalesce(a, b):
    return a.where(~a.isna(), b)

def _fix_cols_ativos(df: pd.DataFrame) -> pd.DataFrame:
    """
    Garante que df_ativos tenha:
      - colunas únicas (sem duplicadas)
      - 'Cargo' e 'Cargo Especifico' separados (se vierem duplicados como 'Cargo Especifico')
    Regra aplicada:
      - Se houver 2+ 'Cargo Especifico' e não existir 'Cargo', renomeia a 1ª ocorrência para 'Cargo'
      - Se houver só 1 'Cargo Especifico' e não existir 'Cargo', cria 'Cargo' copiando 'Cargo Especifico'
    """
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()

    cols = pd.Series(df.columns)
    mask_ce = cols.eq("Cargo Especifico")

    if "Cargo" not in cols.values:
        if mask_ce.sum() >= 2:
            idxs = np.flatnonzero(mask_ce.values)
            cols.iloc[idxs[0]] = "Cargo"  # 1ª ocorrência vira Cargo
        elif mask_ce.sum() == 1:
            # cria Cargo copiando Cargo Especifico (melhor do que NA)
            df["Cargo"] = df["Cargo Especifico"]

    df.columns = cols.tolist()

    # remove duplicadas (se ainda restar alguma)
    df = df.loc[:, ~df.columns.duplicated(keep="last")]
    return df

def _dedup_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    return df.loc[:, ~df.columns.duplicated(keep="last")]

def sincronizar_desligamentos(df_movimentacoes, df_ativos, df_desligados, verbose=True):
    mov = _dedup_cols(df_movimentacoes)
    ativos = _fix_cols_ativos(df_ativos)          # <-- AQUI resolve "Cargo Especifico" duplicado
    deslig = _dedup_cols(df_desligados)

    # --- Normaliza Cadastro (Int64 compatível com NA) ---
    for df in (mov, ativos, deslig):
        df["Cadastro"] = pd.to_numeric(df["Cadastro"], errors="coerce").astype("Int64")

    # --- Garante que df_desligados tenha todas as colunas alvo ---
    for c in TARGET_COLS:
        if c not in deslig.columns:
            deslig[c] = pd.NA
    deslig = deslig[TARGET_COLS].copy()

    # --- Parse datas existentes do df_desligados ---
    deslig["Data_Adm_dt"] = _to_dt(deslig["Data_Adm"], dayfirst=True)
    deslig["Data_Dem_dt"] = _to_dt(deslig["Data_Dem"], dayfirst=True)

    # --- Filtra desligamentos nas movimentações ---
    mov["Tipo da Movimentação"] = mov["Tipo da Movimentação"].astype(str).str.strip()
    mov["Data de alteração_dt"] = _to_dt(mov["Data de alteração"], dayfirst=True)

    mov_desl = mov[mov["Tipo da Movimentação"].str.contains("deslig", case=False, na=False)].copy()

    if mov_desl.empty:
        if verbose:
            print("ℹ️ Nenhum 'Desligamento' encontrado em df_movimentacoes.")
        deslig = deslig.drop(columns=["Data_Adm_dt", "Data_Dem_dt"], errors="ignore")
        return ativos, deslig

    cols_mov_util = ["Cadastro", "Data de alteração_dt"]

    if "Cargo" in mov_desl.columns:
        cols_mov_util.append("Cargo")

    for cand in ["Abreviação UO", "Abreviação UO antiga"]:
        if cand in mov_desl.columns:
            cols_mov_util.append(cand); break
    for cand in ["Desc. UO", "Desc. UO antiga"]:
        if cand in mov_desl.columns:
            cols_mov_util.append(cand); break
    for cand in ["Desc. Cargo", "Desc. Cargo antigo"]:
        if cand in mov_desl.columns:
            cols_mov_util.append(cand); break

    mov_last = (
        mov_desl.sort_values("Data de alteração_dt")
               .groupby("Cadastro", as_index=False)
               .tail(1)[cols_mov_util]
               .rename(columns={"Data de alteração_dt": "Data_Dem_dt"})
    )

    incoming = pd.DataFrame({"Cadastro": mov_last["Cadastro"]})
    incoming["Cargo"] = mov_last["Cargo"] if "Cargo" in mov_last.columns else pd.NA

    incoming["Nome completo"] = pd.NA
    incoming["Heranca"] = pd.NA
    incoming["Local"] = pd.NA
    incoming["Data_Adm_dt"] = pd.NaT
    incoming["Data_Dem_dt"] = mov_last["Data_Dem_dt"]

    col_abrev = next((c for c in ["Abreviação UO", "Abreviação UO antiga"] if c in mov_last.columns), None)
    col_uo = next((c for c in ["Desc. UO", "Desc. UO antiga"] if c in mov_last.columns), None)
    col_cargo = next((c for c in ["Desc. Cargo", "Desc. Cargo antigo"] if c in mov_last.columns), None)

    incoming["Área"] = mov_last[col_abrev] if col_abrev else pd.NA
    incoming["Departamento"] = mov_last[col_uo] if col_uo else pd.NA
    incoming["Cargo Especifico"] = mov_last[col_cargo] if col_cargo else pd.NA

    ativos["Data_Admissão_dt"] = _to_dt(ativos.get("Data_Admissão"), dayfirst=True)

    in_ativos = incoming.merge(ativos, on="Cadastro", how="left", suffixes=("", "_a"))

    # (opcional mas recomendado) puxar também o Cargo de ativos, se veio NA em mov
    incoming["Cargo"] = _coalesce(incoming["Cargo"], in_ativos.get("Cargo_a"))

    incoming["Nome completo"] = _coalesce(incoming["Nome completo"], in_ativos.get("Nome completo_a"))
    incoming["Heranca"] = _coalesce(incoming["Heranca"], in_ativos.get("Heranca_a"))
    incoming["Cargo Especifico"] = _coalesce(incoming["Cargo Especifico"], in_ativos.get("Cargo Especifico_a"))
    incoming["Departamento"] = _coalesce(incoming["Departamento"], in_ativos.get("Departamento_a"))
    incoming["Área"] = _coalesce(incoming["Área"], in_ativos.get("Área_a"))
    incoming["Local"] = _coalesce(incoming["Local"], in_ativos.get("Local_a"))
    incoming["Data_Adm_dt"] = _coalesce(incoming["Data_Adm_dt"], in_ativos.get("Data_Admissão_dt"))

    cad_mover = incoming["Cadastro"].dropna().unique()
    ativos_out = ativos[~ativos["Cadastro"].isin(cad_mover)].reset_index(drop=True)

    base = deslig.copy()

    merged = base.merge(
        incoming,
        on="Cadastro",
        how="outer",
        suffixes=("_old", "_new")
    )

    for col in ["Cargo", "Nome completo", "Heranca", "Cargo Especifico", "Departamento", "Área", "Local"]:
        merged[col] = _coalesce(merged.get(f"{col}_old"), merged.get(f"{col}_new"))

    merged["Data_Adm_dt"] = _coalesce(merged.get("Data_Adm_dt_old"), merged.get("Data_Adm_dt_new"))

    old_dem = merged.get("Data_Dem_dt_old")
    new_dem = merged.get("Data_Dem_dt_new")
    merged["Data_Dem_dt"] = old_dem
    merged.loc[merged["Data_Dem_dt"].isna(), "Data_Dem_dt"] = new_dem
    both = old_dem.notna() & new_dem.notna()
    merged.loc[both, "Data_Dem_dt"] = pd.concat([old_dem[both], new_dem[both]], axis=1).max(axis=1)

    merged["Data_Adm"] = pd.to_datetime(merged["Data_Adm_dt"], errors="coerce").dt.strftime("%d.%m.%Y")
    merged["Data_Dem"] = pd.to_datetime(merged["Data_Dem_dt"], errors="coerce").dt.strftime("%d.%m.%Y")

    desligados_out = merged[TARGET_COLS].reset_index(drop=True)

    if verbose:
        print(f"🔎 Desligamentos (movimentações): {len(incoming)}")
        removidos = df_ativos["Cadastro"].isin(cad_mover).sum() if "Cadastro" in df_ativos.columns else 0
        print(f"🧹 Removidos de ativos (se existiam): {removidos}")
        print(f"✅ Ativos: {len(df_ativos)} -> {len(ativos_out)}")
        print(f"✅ Desligados: {len(df_desligados)} -> {len(desligados_out)}")

    ativos_out = ativos_out.drop(columns=["Data_Admissão_dt"], errors="ignore")
    return ativos_out, desligados_out


In [ ]:
# Executando a Função:
df_ativos, df_desligados = sincronizar_desligamentos(
    df_movimentacoes, df_ativos, df_desligados, verbose=True
)


In [ ]:
# Normalização mínima do 'Cadastro' (evita 123, "123", 123.0 divergirem)
def _norm_id(s):
    s = s.astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    return s.replace({'nan': np.nan})  # volta NaN onde era NaN

for _df in (df_ronda, df_desligados, df_ativos):
    if 'Cadastro' not in _df.columns:
        raise KeyError("Faltou a coluna 'Cadastro' em um dos DataFrames.")
    _df['Cadastro'] = _norm_id(_df['Cadastro'])

# Desligado por Cadastro:
df_ronda['Desligado'] = df_ronda['Cadastro'].isin(df_desligados['Cadastro']).map({True: 'Sim', False: 'Não'})

# Preparar 'df_ativos' com nomes únicos de coluna:
deps = df_ativos[['Cadastro','Nome completo','Área','Departamento','Heranca','Cargo Especifico']].rename(
    columns={
        'Nome completo':'Nome_dep',
        'Área':'Área_dep',
        'Departamento':'Departamento_dep',
        'Heranca':'Heranca_dep',
        'Cargo Especifico':'Cargo Especifico_dep'
    }
)


In [ ]:
# Merge (left) pelo Cadastro:
df_ronda = df_ronda.merge(deps, on='Cadastro', how='left')


In [ ]:
# Dicionários (df_desligados) por Cadastro:
map_df_desligados_nome = dict(zip(df_desligados['Cadastro'], df_desligados['Nome completo']))
map_df_desligados_area = dict(zip(df_desligados['Cadastro'], df_desligados['Área']))
map_df_desligados_dept = dict(zip(df_desligados['Cadastro'], df_desligados['Departamento']))
map_df_desligados_her  = dict(zip(df_desligados['Cadastro'], df_desligados['Heranca']))
map_df_desligados_carg = dict(zip(df_desligados['Cadastro'], df_desligados['Cargo Especifico']))

# Substituir com prioridade: df_desligados > df_ativos > original:

# Nome:
df_ronda['Nome'] = (
    df_ronda['Cadastro'].map(map_df_desligados_nome)   # 1º: df_desligados…
    .fillna(df_ronda['Nome_dep'])                  # 2º: df_ativos…
    .fillna(df_ronda.get('Nome'))                  # 3º: valor original (se existir)…
)

# Área:
df_ronda['Área'] = (
    df_ronda['Cadastro'].map(map_df_desligados_area)
    .fillna(df_ronda['Área_dep'])
)

# Departamento:
df_ronda['Departamento'] = (
    df_ronda['Cadastro'].map(map_df_desligados_dept)
    .fillna(df_ronda['Departamento_dep'])
)

# Heranca:
df_ronda['Heranca'] = (
    df_ronda['Cadastro'].map(map_df_desligados_her)
    .fillna(df_ronda['Heranca_dep'])
)

# Cargo Especifico:
df_ronda['Cargo Especifico'] = (
    df_ronda['Cadastro'].map(map_df_desligados_carg)
    .fillna(df_ronda['Cargo Especifico_dep'])
)

# (Opcional) Manter as colunas *_dep para auditoria; se quiser remover, descomente abaixo:
df_ronda = df_ronda.drop(columns=['Nome_dep','Área_dep','Departamento_dep','Heranca_dep','Cargo Especifico_dep'], errors='ignore')

#### Corrigindo as colunas de Datas:

In [ ]:
# Pareseand por regex:
s = df_ronda["Data"].astype(str).str.strip()

# só datas com barra (ex: 02/09/2026)
mask_slash = s.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$")

# tenta BR primeiro (dd/mm/yyyy) sem warning, porque o format é explícito
br = pd.to_datetime(s.where(mask_slash), format="%d/%m/%Y", errors="coerce")

# onde falhou, tenta US (mm/dd/yyyy) também sem warning
us = pd.to_datetime(s.where(mask_slash), format="%m/%d/%Y", errors="coerce")

# para outros formatos (ex: YYYY-MM-DD), tenta o parser padrão
other = pd.to_datetime(s.where(~mask_slash), errors="coerce")

df_ronda["Data"] = other.fillna(br.fillna(us))


In [ ]:
# Hora (minutos -> time):
df_ronda['Hora'] = pd.to_numeric(df_ronda['Hora'], errors='coerce')
df_ronda['Hora'] = pd.to_timedelta(df_ronda['Hora'], unit='m')
df_ronda['Hora'] = (pd.Timestamp('1900-01-01') + df_ronda['Hora']).dt.time

# Data (MANTER como datetime — não usar strftime aqui):
df_ronda['Data'] = pd.to_datetime(df_ronda['Data'], dayfirst=True, errors='coerce')

# Admissão/Demissão (datetime):
df_desligados['Data_Adm'] = pd.to_datetime(df_desligados['Data_Adm'], dayfirst=True, errors='coerce')
df_desligados['Data_Dem'] = pd.to_datetime(df_desligados['Data_Dem'], dayfirst=True, errors='coerce')
df_ativos['Data_Admissão'] = pd.to_datetime(df_ativos['Data_Admissão'], format='%d.%m.%Y', errors='coerce')

def fmt_dt(x):
    return x.strftime('%d/%m/%Y') if pd.notna(x) else 'NaT'

print(f"df_ronda['Data']           -> min={fmt_dt(df_ronda['Data'].min())} | max={fmt_dt(df_ronda['Data'].max())}")
print(f"df_desligados['Data_Adm']  -> min={fmt_dt(df_desligados['Data_Adm'].min())} | max={fmt_dt(df_desligados['Data_Adm'].max())}")
print(f"df_desligados['Data_Dem']  -> min={fmt_dt(df_desligados['Data_Dem'].min())} | max={fmt_dt(df_desligados['Data_Dem'].max())}")
print(f"df_ativos['Data_Admissão'] -> min={fmt_dt(df_ativos['Data_Admissão'].min())} | max={fmt_dt(df_ativos['Data_Admissão'].max())}")

## 3. Carregando os Dados:

### Desenvolvendo toda a Engenharia dos Dados em formato de Data Warehouse:

In [ ]:
# ETL "executável" (sem função principal) para construir o DW SQLite `dw_ronda.sqlite` a partir do df_ronda.
"""
✅ Foco: performance no APP (Dash) via tabelas agregadas MATERIALIZADAS (agg_*).
✅ O app abre o arquivo em READ-ONLY / immutable, sem DDL.

Pré-requisitos no Notebook:
  - df_ronda (obrigatório)
  - df_movimentacoes (opcional)
  - df_gptw (opcional)
  - df_headcount (opcional)  # se não passar, tenta calcular via df_ativos/df_desligados; senão usa fallback por presença
  - df_ativos (opcional; RH)   # snapshot atual (Cadastro, Área, Heranca, Data_Admissão)
  - df_desligados (opcional; RH)  # histórico (Cadastro, Área, Heranca, Data_Adm, Data_Dem)

Como usar (no fim do arquivo tem um bloco "EXECUÇÃO"):
  - ajuste SQLITE_PATH e CHUNK_SIZE
  - garanta que df_ronda exista em memória
  - rode o script/célula

Obs: Datas ficam como TEXT ISO (YYYY-MM-DD) no SQLite (ótimo p/ índice). No app, parse com pd.to_datetime.
"""

# =========================
# CONFIG (AJUSTE AQUI)
# =========================
SQLITE_PATH = Path(r"..\project\database\dw_ronda.sqlite")
CHUNK_SIZE = 250_000  # ajuste conforme RAM

# Otimizações (tuning)
FAST_ETL_PRAGMAS = True   # acelera build (recomendado; o DW é reconstruível)
DO_ANALYZE = True        # gera estatísticas p/ planner
DO_OPTIMIZE = True       # PRAGMA optimize (SQLite 3.18+)
DO_VACUUM = False        # compacta arquivo (custo alto no ETL)


# =========================
# HELPERS
# =========================
def normalize_columns_inplace(df: pd.DataFrame) -> None:
    df.columns = [c.strip() for c in df.columns]
    rename_map = {
        "Cadastro": "cadastro",
        "Nome": "nome",
        "Filial": "filial",
        "Data": "data",
        "Hora": "hora",
        "Dir": "dir",
        "Tipo": "tipo",
        "Acesso": "acesso",
        "Planta": "planta",
        "Coletor": "coletor",
        "Descricao Coletor": "descricao_coletor",
        "Descrição Coletor": "descricao_coletor",
        "Uso Acesso": "uso_acesso",
        "Cargo": "cargo",
        "Desligado": "desligado",
        "Área": "area",
        "Area": "area",
        "Departamento": "departamento",
        "Heranca": "heranca",
        "Herança": "heranca",
        "Cargo Especifico": "cargo_especifico",
        "Cargo Específico": "cargo_especifico",
    }
    df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)

def _clean_date_time(chunk: pd.DataFrame) -> tuple[pd.Series, pd.Series, pd.Series]:
    # data_iso
    data_dt = pd.to_datetime(chunk["data"], errors="coerce")
    data_iso = pd.Series(data_dt.dt.strftime("%Y-%m-%d"), dtype="string")

    # hora_iso (extrai HH:MM(:SS)? e normaliza)
    hora_str = chunk["hora"].astype("string").str.strip()
    hora_extr = hora_str.str.extract(r"(\d{1,2}:\d{2}(?::\d{2})?)", expand=False)
    hora_clean = hora_extr.where(
        hora_extr.str.contains(r":\d{2}:\d{2}$", regex=True),
        hora_extr + ":00",
    )
    hora_clean = hora_clean.str.replace(r"^(\d):", r"0\1:", regex=True)
    hora_dt = pd.to_datetime(hora_clean, format="%H:%M:%S", errors="coerce")
    hora_iso = pd.Series(hora_dt.dt.strftime("%H:%M:%S"), dtype="string")

    # ts_iso
    ts = pd.to_datetime((data_iso + " " + hora_iso), format="%Y-%m-%d %H:%M:%S", errors="coerce")
    ts_iso = pd.Series(ts.dt.strftime("%Y-%m-%d %H:%M:%S"), dtype="string")

    return data_iso, hora_iso, ts_iso

# =========================
# HEADCOUNT (HR) - helpers (roll-up por Heranca)
# =========================
import re

def _hc_norm(x):
    return ("" if pd.isna(x) else str(x).strip().upper())

def _hc_to_dt(s):
    return pd.to_datetime(s, errors="coerce", dayfirst=True)

def _hc_rename_heranca(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df, pd.DataFrame) and "Herança" in df.columns and "Heranca" not in df.columns:
        return df.rename(columns={"Herança": "Heranca"})
    return df

def _hc_area_clean(area: str) -> str:
    # remove tudo que não é letra/número -> base p/ prefixo
    return re.sub(r"[^A-Z0-9]", "", _hc_norm(area))

def _hc_map_area_to_heranca(df_ativos: pd.DataFrame, df_desligados: pd.DataFrame) -> dict:
    frames = []
    for df in (_hc_rename_heranca(df_ativos), _hc_rename_heranca(df_desligados)):
        if isinstance(df, pd.DataFrame) and {"Área", "Heranca"}.issubset(df.columns):
            t = df[["Área", "Heranca"]].copy()
            t["Área"] = t["Área"].map(_hc_norm)
            t["Heranca"] = t["Heranca"].map(_hc_norm)
            t = t[(t["Área"] != "") & (t["Heranca"] != "") & (t["Heranca"] != "****")]
            frames.append(t)
    if not frames:
        return {}
    cat = pd.concat(frames, ignore_index=True)
    # moda por área (estável o suficiente)
    return cat.groupby("Área")["Heranca"].agg(
        lambda s: s.mode().iat[0] if not s.mode().empty else ""
    ).to_dict()

def _hc_area_parents(area: str, areaher: dict) -> list[str]:
    area_n = _hc_norm(area)
    clean = _hc_area_clean(area_n)
    her = _hc_norm(areaher.get(area_n, ""))
    if not clean or not her:
        return []

    parents = []
    for cand, h in (areaher or {}).items():
        if _hc_norm(h) != her:
            continue
        cand_n = _hc_norm(cand)
        cand_clean = _hc_area_clean(cand_n)
        if len(cand_clean) < len(clean) and clean.startswith(cand_clean):
            parents.append(cand_n)

    # mais próximo primeiro
    parents = sorted(set(parents), key=lambda a: len(_hc_area_clean(a)), reverse=True)
    return parents

def _hc_rollup_full_counts(full_counts: dict, areaher: dict) -> dict:
    out = dict(full_counts or {})
    for a_full, n in (full_counts or {}).items():
        a_full_n = _hc_norm(a_full)
        for p in _hc_area_parents(a_full_n, areaher):
            out[p] = out.get(p, 0) + int(n)
    return out

def _hc_mov_id_col(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    norm = {c: _hc_norm(c) for c in cols}
    exact = {"CADASTRO", "Nº PESSOAL", "NO PESSOAL", "NUM PESSOAL", "ID", "MATRICULA", "MATRÍCULA"}
    for c, n in norm.items():
        if n in exact:
            return c
    for c, n in norm.items():
        if any(k in n for k in ["CADASTRO", "PESSOAL", "MATRIC"]):
            return c
    raise KeyError(f"Não encontrei coluna de ID em df_movimentacoes. Colunas: {cols}")

def _hc_consolida_bases(df_ativos: pd.DataFrame, df_desligados: pd.DataFrame) -> pd.DataFrame:
    # df_ativos (snapshot atual)
    dep = _hc_rename_heranca(df_ativos).copy()
    if not dep.empty:
        dep["Cadastro"] = dep["Cadastro"].astype(str)
        dep["Área"] = dep.get("Área", "").map(_hc_norm) if "Área" in dep.columns else ""
        dep["Heranca"] = dep.get("Heranca", "").map(_hc_norm) if "Heranca" in dep.columns else ""
        dep["Data_Admissão"] = _hc_to_dt(dep.get("Data_Admissão")) if "Data_Admissão" in dep.columns else pd.NaT

        dep_cur = (
            dep.sort_values(["Cadastro", "Data_Admissão"])
               .drop_duplicates("Cadastro", keep="last")
               .rename(columns={"Área": "Área_cur", "Heranca": "Her_cur", "Data_Admissão": "Adm_dep"})
        )[["Cadastro", "Área_cur", "Her_cur", "Adm_dep"]]
    else:
        dep_cur = pd.DataFrame(columns=["Cadastro", "Área_cur", "Her_cur", "Adm_dep"])

    # df_desligados (histórico)
    afa = _hc_rename_heranca(df_desligados).copy()
    if not afa.empty:
        afa["Cadastro"] = afa["Cadastro"].astype(str)
        afa["Área"] = afa.get("Área", "").map(_hc_norm) if "Área" in afa.columns else ""
        afa["Heranca"] = afa.get("Heranca", "").map(_hc_norm) if "Heranca" in afa.columns else ""
        afa["Data_Adm"] = _hc_to_dt(afa.get("Data_Adm")) if "Data_Adm" in afa.columns else pd.NaT
        afa["Data_Dem"] = _hc_to_dt(afa.get("Data_Dem")) if "Data_Dem" in afa.columns else pd.NaT

        # primeira admissão
        adm_first = (
            afa.sort_values(["Cadastro", "Data_Adm"])
               .dropna(subset=["Data_Adm"])
               .drop_duplicates("Cadastro", keep="first")
               .rename(columns={"Área": "Área_on_adm", "Heranca": "Her_on_adm", "Data_Adm": "Adm_first"})
        )[["Cadastro", "Área_on_adm", "Her_on_adm", "Adm_first"]]

        # última demissão
        dem_last = (
            afa.sort_values(["Cadastro", "Data_Dem"])
               .dropna(subset=["Data_Dem"])
               .drop_duplicates("Cadastro", keep="last")
               .rename(columns={"Data_Dem": "Dem_last"})
        )[["Cadastro", "Dem_last"]]
    else:
        adm_first = pd.DataFrame(columns=["Cadastro", "Área_on_adm", "Her_on_adm", "Adm_first"])
        dem_last = pd.DataFrame(columns=["Cadastro", "Dem_last"])

    base = dep_cur.merge(adm_first, on="Cadastro", how="outer").merge(dem_last, on="Cadastro", how="outer")

    base["Adm_global"] = base[["Adm_dep", "Adm_first"]].min(axis=1)
    base["Dem_global"] = base["Dem_last"]

    base["Área_init"] = base["Área_on_adm"].where(
        base["Área_on_adm"].notna() & (base["Área_on_adm"] != ""),
        base["Área_cur"],
    )

    return base

def _hc_area_asof(dt: pd.Timestamp, base: pd.DataFrame, df_movimentacoes: pd.DataFrame | None) -> pd.Series:
    # filtra ativos na data
    asof = base[["Cadastro", "Área_init", "Adm_global", "Dem_global"]].copy()
    asof = asof[
        (asof["Adm_global"].notna())
        & (asof["Adm_global"] <= dt)
        & (asof["Dem_global"].isna() | (asof["Dem_global"] > dt))
    ].copy()

    if df_movimentacoes is None or not isinstance(df_movimentacoes, pd.DataFrame) or df_movimentacoes.empty:
        return asof.set_index("Cadastro")["Área_init"].astype(str).map(_hc_norm)

    mv = df_movimentacoes.copy()
    idcol = _hc_mov_id_col(mv)
    mv[idcol] = mv[idcol].astype(str)

    # normaliza
    for c in ("Abreviação UO", "Abreviação UO antiga"):
        if c in mv.columns:
            mv[c] = mv[c].map(_hc_norm)
    mv["Data de alteração"] = _hc_to_dt(mv.get("Data de alteração"))

    mv = mv[
        mv["Data de alteração"].notna()
        & (mv.get("Abreviação UO", "") != "")
        & (mv.get("Abreviação UO antiga", "") != "")
    ].sort_values([idcol, "Data de alteração"])

    # origem histórica: primeira área antiga
    earliest_old = mv.groupby(idcol)["Abreviação UO antiga"].first().rename("Área_earliest_old")

    # posição vigente: última área nova até a data
    mv_upto = mv[mv["Data de alteração"] <= dt]
    last_new = mv_upto.groupby(idcol)["Abreviação UO"].last().rename("Área_last_upto")

    eff = asof.merge(earliest_old, left_on="Cadastro", right_index=True, how="left")
    eff = eff.merge(last_new, left_on="Cadastro", right_index=True, how="left")

    eff["Área_asof"] = eff["Área_last_upto"].where(
        eff["Área_last_upto"].notna() & (eff["Área_last_upto"] != ""),
        eff["Área_earliest_old"].where(
            eff["Área_earliest_old"].notna() & (eff["Área_earliest_old"] != ""),
            eff["Área_init"],
        ),
    )

    return eff.set_index("Cadastro")["Área_asof"].astype(str).map(_hc_norm)

def _hc_snapshot_from_df_ativos(df_ativos: pd.DataFrame, areaher: dict) -> pd.Series:
    dep = _hc_rename_heranca(df_ativos).copy()
    if dep.empty or "Área" not in dep.columns or "Cadastro" not in dep.columns:
        return pd.Series(dtype=int)

    dep["Cadastro"] = dep["Cadastro"].astype(str)
    dep["Área"] = dep["Área"].map(_hc_norm)
    dep = dep[dep["Área"] != ""].copy()

    if "Data_Admissão" in dep.columns:
        dep["Data_Admissão"] = _hc_to_dt(dep["Data_Admissão"])
        dep = dep.sort_values(["Cadastro", "Data_Admissão"]).drop_duplicates("Cadastro", keep="last")
    else:
        dep = dep.drop_duplicates("Cadastro", keep="last")

    full_counts = dep["Área"].value_counts().to_dict()
    rolled = _hc_rollup_full_counts(full_counts, areaher)
    return pd.Series(rolled, dtype=int)

def compute_df_headcount_from_hr(
    df_ativos: pd.DataFrame,
    df_desligados: pd.DataFrame,
    df_movimentacoes: pd.DataFrame | None,
    anos=(2023, 2024, 2025),
) -> pd.DataFrame:
    # mapa área -> herança (pra roll-up)
    areaher = _hc_map_area_to_heranca(df_ativos, df_desligados)

    # base consolidada
    base = _hc_consolida_bases(df_ativos, df_desligados)

    # headcount por ano via snapshot 31/12 (as-of) + roll-up
    hc_year = {}
    for y in anos:
        dt = pd.Timestamp(int(y), 12, 31)
        area_asof = _hc_area_asof(dt, base, df_movimentacoes).dropna().astype(str).map(_hc_norm)
        full_counts = area_asof.value_counts().to_dict()
        hc_year[int(y)] = pd.Series(_hc_rollup_full_counts(full_counts, areaher), dtype=int)

    # ancoragem opcional do último ano via df_ativos (mantém sua lógica)
    if anos:
        last_year = int(max(anos))
        snap = _hc_snapshot_from_df_ativos(df_ativos, areaher)
        if isinstance(snap, pd.Series) and not snap.empty:
            hc_year[last_year] = snap.astype(int)

    # catálogo final de áreas (inclui subáreas e pais)
    all_areas = set()
    for s in hc_year.values():
        all_areas |= set(map(str, s.index))
    all_areas |= set(map(str, (areaher or {}).keys()))
    # garante pais no catálogo
    for a in list(all_areas):
        for p in _hc_area_parents(a, areaher):
            all_areas.add(p)

    areas = sorted({ _hc_norm(a) for a in all_areas if _hc_norm(a) })

    # monta tabela area x ano
    out = pd.DataFrame({"area": areas})
    for y in (2023, 2024, 2025):
        s = hc_year.get(y, pd.Series(dtype=int)).reindex(areas).fillna(0).astype(int)
        out[f"y{y}"] = s.values

    return out[["area", "y2023", "y2024", "y2025"]]


# =========================
# DDL (MATERIALIZADO)
# =========================
DDL_SQL = r"""
PRAGMA foreign_keys = ON;

-- Base (flat)
DROP TABLE IF EXISTS evento_acesso_flat;
CREATE TABLE evento_acesso_flat (
  cadastro            TEXT,
  data                TEXT,  -- YYYY-MM-DD
  hora                TEXT,  -- HH:MM:SS
  ts                  TEXT,  -- YYYY-MM-DD HH:MM:SS
  filial              TEXT,
  planta              TEXT,
  coletor             TEXT,  -- UPPER no ETL
  descricao_coletor   TEXT,
  dir                 TEXT,  -- UPPER no ETL
  tipo                TEXT,
  acesso              TEXT,
  uso_acesso          TEXT
);

-- Dim colaborador (último estado por cadastro)
DROP TABLE IF EXISTS dim_colaborador;
CREATE TABLE dim_colaborador (
  cadastro            TEXT PRIMARY KEY,
  nome                TEXT,
  cargo               TEXT,
  cargo_especifico    TEXT,
  area                TEXT,
  departamento        TEXT,
  heranca             TEXT,
  desligado           TEXT,
  created_at          TEXT DEFAULT (datetime('now'))
) WITHOUT ROWID;

-- Agregados p/ app
DROP TABLE IF EXISTS agg_diarios_unico;
CREATE TABLE agg_diarios_unico (
  cadastro TEXT,
  data     TEXT,
  planta   TEXT,
  PRIMARY KEY (cadastro, data)
) WITHOUT ROWID;

DROP TABLE IF EXISTS agg_registros_semanais;
CREATE TABLE agg_registros_semanais (
  cadastro TEXT,
  ano      INTEGER,
  semana_por_ano INTEGER,
  data     TEXT,
  contagem_presenca INTEGER,
  PRIMARY KEY (cadastro, ano, semana_por_ano)
) WITHOUT ROWID;

DROP TABLE IF EXISTS agg_semana_enriquecida;
CREATE TABLE agg_semana_enriquecida (
  cadastro TEXT,
  nome     TEXT,
  cargo    TEXT,
  cargo_especifico TEXT,
  area     TEXT,
  departamento TEXT,
  heranca  TEXT,
  desligado TEXT,
  ano      INTEGER,
  semana_por_ano INTEGER,
  data     TEXT,
  contagem_presenca INTEGER
);

DROP TABLE IF EXISTS agg_almoco;
CREATE TABLE agg_almoco (
  cadastro TEXT,
  data     TEXT,
  hora     TEXT,
  dir      TEXT,
  planta   TEXT,
  coletor  TEXT,
  descricao_coletor TEXT
);

-- Opcionais
DROP TABLE IF EXISTS gptw;
CREATE TABLE gptw (area TEXT, y2025 REAL, y2024 REAL, y2023 REAL);

DROP TABLE IF EXISTS movimentacoes;
CREATE TABLE movimentacoes (
  cadastro TEXT,
  abreviacao_uo_antiga TEXT,
  desc_uo_antiga TEXT,
  desc_cargo_antigo TEXT,
  abreviacao_uo TEXT,
  desc_uo TEXT,
  desc_cargo TEXT,
  tipo_movimentacao TEXT,
  data_alteracao TEXT
);

DROP TABLE IF EXISTS headcount;
CREATE TABLE headcount (area TEXT, y2023 INTEGER, y2024 INTEGER, y2025 INTEGER);

-- Views PT-BR (contratos do app)
DROP VIEW IF EXISTS vw_colaboradores;
CREATE VIEW vw_colaboradores AS
SELECT
  cadastro            AS "Cadastro",
  nome                AS "Nome",
  cargo               AS "Cargo",
  cargo_especifico    AS "Cargo Especifico",
  area                AS "Área",
  departamento        AS "Departamento",
  heranca             AS "Heranca",
  desligado           AS "Desligado"
FROM dim_colaborador;

DROP VIEW IF EXISTS vw_df_diarios_unico;
CREATE VIEW vw_df_diarios_unico AS
SELECT cadastro AS "Cadastro", planta AS "Planta", data AS "Data"
FROM agg_diarios_unico;

DROP VIEW IF EXISTS vw_df_diarios;
CREATE VIEW vw_df_diarios AS
SELECT cadastro AS "Cadastro", planta AS "Planta", data AS "Data"
FROM agg_diarios_unico;

DROP VIEW IF EXISTS vw_df_registros_semanais;
CREATE VIEW vw_df_registros_semanais AS
SELECT
  cadastro AS "Cadastro",
  ano AS "Ano",
  semana_por_ano AS "Semana por Ano",
  data AS "Data",
  contagem_presenca AS "Contagem Presença"
FROM agg_registros_semanais;

DROP VIEW IF EXISTS vw_semana_enriquecida;
CREATE VIEW vw_semana_enriquecida AS
SELECT
  cadastro AS "Cadastro",
  nome AS "Nome",
  cargo AS "Cargo",
  cargo_especifico AS "Cargo Especifico",
  area AS "Área",
  departamento AS "Departamento",
  heranca AS "Heranca",
  desligado AS "Desligado",
  ano AS "Ano",
  semana_por_ano AS "Semana por Ano",
  data AS "Data",
  contagem_presenca AS "Contagem Presença"
FROM agg_semana_enriquecida;

DROP VIEW IF EXISTS vw_df_almoco;
CREATE VIEW vw_df_almoco AS
SELECT
  cadastro AS "Cadastro",
  data AS "Data",
  hora AS "Hora",
  dir AS "Dir",
  planta AS "Planta",
  coletor AS "Coletor",
  descricao_coletor AS "Descricao Coletor"
FROM agg_almoco;

DROP VIEW IF EXISTS vw_gptw;
CREATE VIEW vw_gptw AS
SELECT area AS "Área", CAST(y2025 AS REAL) AS "2025", CAST(y2024 AS REAL) AS "2024", CAST(y2023 AS REAL) AS "2023"
FROM gptw;

DROP VIEW IF EXISTS vw_movimentacoes;
CREATE VIEW vw_movimentacoes AS
SELECT
  cadastro AS "Cadastro",
  abreviacao_uo_antiga AS "Abreviação UO antiga",
  desc_uo_antiga AS "Desc. UO antiga",
  desc_cargo_antigo AS "Desc. Cargo antigo",
  abreviacao_uo AS "Abreviação UO",
  desc_uo AS "Desc. UO",
  desc_cargo AS "Desc. Cargo",
  tipo_movimentacao AS "Tipo da Movimentação",
  data_alteracao AS "Data de alteração"
FROM movimentacoes;

DROP VIEW IF EXISTS vw_headcount;
CREATE VIEW vw_headcount AS
SELECT area AS "Área", CAST(y2023 AS INTEGER) AS "2023", CAST(y2024 AS INTEGER) AS "2024", CAST(y2025 AS INTEGER) AS "2025"
FROM headcount;
"""

# =========================
# EXECUÇÃO (SEM FUNÇÃO)
# =========================

# Espera-se que df_ronda exista no ambiente.
# df_movimentacoes/df_gptw/df_headcount são opcionais; se não existirem, deixa como None.
df_movimentacoes = globals().get("df_movimentacoes", None)
df_gptw = globals().get("df_gptw", None)
df_headcount = globals().get("df_headcount", None)

assert "df_ronda" in globals(), "df_ronda não encontrado no ambiente. Carregue df_ronda antes de rodar o ETL."
df_ronda = globals()["df_ronda"]

SQLITE_PATH.parent.mkdir(parents=True, exist_ok=True)
if SQLITE_PATH.exists():
    SQLITE_PATH.unlink()

normalize_columns_inplace(df_ronda)

required = [
    "cadastro", "nome", "filial", "data", "hora", "dir", "tipo", "acesso", "planta",
    "coletor", "descricao_coletor", "uso_acesso", "cargo", "desligado", "area",
    "departamento", "heranca", "cargo_especifico",
]

# Para DIM: manter “último estado” por cadastro sem guardar 35M linhas em staging
# Guardamos em dict: cadastro -> (ts, nome, cargo, cargo_especifico, area, departamento, heranca, desligado)
dim_latest: dict[str, tuple[str, str, str, str, str, str, str, str]] = {}

conn = sqlite3.connect(SQLITE_PATH)
try:
    # Bulk load PRAGMAs
    conn.execute("PRAGMA page_size = 32768;")
    conn.execute("PRAGMA journal_mode = OFF;")
    conn.execute("PRAGMA synchronous = OFF;")
    conn.execute("PRAGMA temp_store = MEMORY;")
    conn.execute("PRAGMA cache_size = -600000;")  # ~600MB
    conn.execute("PRAGMA locking_mode = EXCLUSIVE;")
    conn.execute("PRAGMA foreign_keys = ON;")

    conn.executescript(DDL_SQL)
    cur = conn.cursor()

    insert_flat = """INSERT INTO evento_acesso_flat (
        cadastro,data,hora,ts,filial,planta,coletor,descricao_coletor,dir,tipo,acesso,uso_acesso
    ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?);"""

    n = len(df_ronda)
    for start in range(0, n, CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, n)
        chunk = df_ronda.iloc[start:end].reindex(columns=required, fill_value=pd.NA).copy()
        chunk.replace({"nan": pd.NA, "NaN": pd.NA, "None": pd.NA, "": pd.NA}, inplace=True)

        data_iso, hora_iso, ts_iso = _clean_date_time(chunk)

        # Flat (mínimo necessário p/ agregações + filtros)
        flat_df = pd.DataFrame({
            "cadastro": chunk["cadastro"].astype("string").str.strip(),
            "data": data_iso,
            "hora": hora_iso,
            "ts": ts_iso,
            "filial": chunk["filial"].astype("string").str.strip(),
            "planta": chunk["planta"].astype("string").str.strip(),
            "coletor": chunk["coletor"].astype("string").str.strip().str.upper(),
            "descricao_coletor": chunk["descricao_coletor"].astype("string").str.strip(),
            "dir": chunk["dir"].astype("string").str.strip().str.upper(),
            "tipo": chunk["tipo"].astype("string").str.strip(),
            "acesso": chunk["acesso"].astype("string").str.strip(),
            "uso_acesso": chunk["uso_acesso"].astype("string").str.strip(),
        })

        # ✅ bind-safe: pandas.NA -> None
        flat_obj = flat_df.astype("object").where(flat_df.notna(), None)

        conn.execute("BEGIN;")
        cur.executemany(insert_flat, flat_obj.itertuples(index=False, name=None))
        conn.commit()

        # Atualiza DIM (último estado por cadastro) usando ts
        # Faz isso em pandas para evitar staging gigante + window function em 35M.
        dim_chunk = pd.DataFrame({
            "cadastro": chunk["cadastro"].astype("string").str.strip(),
            "ts": ts_iso,
            "nome": chunk["nome"].astype("string").str.strip(),
            "cargo": chunk["cargo"].astype("string").str.strip(),
            "cargo_especifico": chunk["cargo_especifico"].astype("string").str.strip(),
            "area": chunk["area"].astype("string").str.strip(),
            "departamento": chunk["departamento"].astype("string").str.strip(),
            "heranca": chunk["heranca"].astype("string").str.strip(),
            "desligado": chunk["desligado"].astype("string").str.strip(),
        }).astype("object").where(lambda d: d.notna(), None)

        # remove cadastros/ts vazios
        dim_chunk = dim_chunk[(dim_chunk["cadastro"].notna()) & (dim_chunk["ts"].notna())]

        # pega o registro de maior ts por cadastro no chunk (ts ISO ordena como string)
        dim_chunk.sort_values(["cadastro", "ts"], inplace=True)
        last_per_cad = dim_chunk.groupby("cadastro", as_index=False).tail(1)

        for row in last_per_cad.itertuples(index=False):
            cad = row.cadastro
            ts = row.ts
            prev = dim_latest.get(cad)
            if prev is None or ts > prev[0]:
                dim_latest[cad] = (ts, row.nome, row.cargo, row.cargo_especifico, row.area, row.departamento, row.heranca, row.desligado)

    # Índices no flat (acelera agregações e filtros)
    conn.executescript("""
        CREATE INDEX IF NOT EXISTS ix_flat_cad_data_ts   ON evento_acesso_flat (cadastro, data, ts);
        CREATE INDEX IF NOT EXISTS ix_flat_data          ON evento_acesso_flat (data);
        CREATE INDEX IF NOT EXISTS ix_flat_planta        ON evento_acesso_flat (planta);
        CREATE INDEX IF NOT EXISTS ix_flat_tipo          ON evento_acesso_flat (tipo);
        CREATE INDEX IF NOT EXISTS ix_flat_dir           ON evento_acesso_flat (dir);
        CREATE INDEX IF NOT EXISTS ix_flat_coletor       ON evento_acesso_flat (coletor);
    """)

    # Carrega DIM no SQLite
    conn.execute("BEGIN;")
    conn.execute("DELETE FROM dim_colaborador;")
    conn.executemany(
        "INSERT INTO dim_colaborador(cadastro,nome,cargo,cargo_especifico,area,departamento,heranca,desligado) VALUES (?,?,?,?,?,?,?,?);",
        [(cad, nome, cargo, cargo_esp, area, depto, her, desl) for cad, (ts, nome, cargo, cargo_esp, area, depto, her, desl) in dim_latest.items()]
    )
    conn.commit()
    conn.executescript("""
        CREATE INDEX IF NOT EXISTS ix_dim_area   ON dim_colaborador (area);
        CREATE INDEX IF NOT EXISTS ix_dim_depto  ON dim_colaborador (departamento);
        CREATE INDEX IF NOT EXISTS ix_dim_heranca ON dim_colaborador (heranca);
    """)

    # =========================
    # AGREGAÇÕES MATERIALIZADAS
    # =========================

    # Presença válida + diários único (último ts do dia por cadastro)
    conn.executescript("""
        DROP TABLE IF EXISTS tmp_presenca_validos;
        CREATE TEMP TABLE tmp_presenca_validos AS
        SELECT
          cadastro,
          data AS data_dia,
          ts,
          planta
        FROM evento_acesso_flat
        WHERE
          cadastro IS NOT NULL
          AND data IS NOT NULL
          AND ts IS NOT NULL
          AND planta IS NOT NULL
          AND planta <> '999'
          AND tipo IS NOT NULL
          AND tipo <> '0';

        CREATE INDEX IF NOT EXISTS ix_tmp_pres_cad_data_ts ON tmp_presenca_validos (cadastro, data_dia, ts);

        -- ts máximo por cadastro+dia
        DROP TABLE IF EXISTS tmp_last;
        DROP TABLE IF EXISTS tmp_last_rid;
        CREATE TEMP TABLE tmp_last AS
        SELECT cadastro, data_dia, MAX(ts) AS ts_max
        FROM tmp_presenca_validos
        GROUP BY cadastro, data_dia;

        CREATE INDEX IF NOT EXISTS ix_tmp_last ON tmp_last (cadastro, data_dia);

        -- Empate no mesmo ts: pega a MAIOR rowid (determinístico) e evita duplicar PK
        DROP TABLE IF EXISTS tmp_last_rid;
        CREATE TEMP TABLE tmp_last_rid AS
        SELECT
          p.cadastro,
          p.data_dia,
          MAX(p.rowid) AS rid
        FROM tmp_presenca_validos p
        JOIN tmp_last l
          ON p.cadastro = l.cadastro
         AND p.data_dia = l.data_dia
         AND p.ts = l.ts_max
        GROUP BY p.cadastro, p.data_dia;

        CREATE INDEX IF NOT EXISTS ix_tmp_last_rid ON tmp_last_rid (cadastro, data_dia);

        INSERT INTO agg_diarios_unico (cadastro, data, planta)
        SELECT
          p.cadastro,
          p.data_dia AS data,
          p.planta
        FROM tmp_presenca_validos p
        JOIN tmp_last_rid r
          ON p.cadastro = r.cadastro
         AND p.data_dia = r.data_dia
         AND p.rowid = r.rid;

        CREATE INDEX IF NOT EXISTS ix_diarios_data_planta_cad ON agg_diarios_unico (data, planta, cadastro);
    """)

    # Semana ISO (ancora quinta)
    conn.executescript("""
        WITH base AS (
          SELECT
            cadastro,
            planta,
            data AS data_dia,
            date(data, '-3 days', 'weekday 4') AS iso_anchor
          FROM agg_diarios_unico
        )
        INSERT INTO agg_registros_semanais (cadastro, ano, semana_por_ano, data, contagem_presenca)
        SELECT
          cadastro,
          CAST(strftime('%Y', iso_anchor) AS INTEGER) AS ano,
          CAST(strftime('%W', iso_anchor) AS INTEGER) + 1 AS semana_por_ano,
          date(iso_anchor, '-3 days') AS data,
          COUNT(*) AS contagem_presenca
        FROM base
        GROUP BY cadastro, ano, semana_por_ano, date(iso_anchor, '-3 days');

        CREATE INDEX IF NOT EXISTS ix_semana_data      ON agg_registros_semanais (data);
        CREATE INDEX IF NOT EXISTS ix_semana_ano_sem_cad ON agg_registros_semanais (ano, semana_por_ano, cadastro);
    """)

    # Semana enriquecida
    conn.executescript("""
        INSERT INTO agg_semana_enriquecida (
          cadastro, nome, cargo, cargo_especifico, area, departamento, heranca, desligado,
          ano, semana_por_ano, data, contagem_presenca
        )
        SELECT
          s.cadastro,
          c.nome,
          c.cargo,
          c.cargo_especifico,
          c.area,
          c.departamento,
          c.heranca,
          c.desligado,
          s.ano,
          s.semana_por_ano,
          s.data,
          s.contagem_presenca
        FROM agg_registros_semanais s
        LEFT JOIN dim_colaborador c
          ON c.cadastro = s.cadastro;

        CREATE INDEX IF NOT EXISTS ix_semana_enr_data_cad       ON agg_semana_enriquecida (data, cadastro);
        CREATE INDEX IF NOT EXISTS ix_semana_enr_area_data     ON agg_semana_enriquecida (area, data);
        CREATE INDEX IF NOT EXISTS ix_semana_enr_depto_data    ON agg_semana_enriquecida (departamento, data);
        CREATE INDEX IF NOT EXISTS ix_semana_enr_heranca_data  ON agg_semana_enriquecida (heranca, data);
    """)

    # Almoço (dir/coletor já UPPER)
    conn.executescript("""
        INSERT INTO agg_almoco (cadastro, data, hora, dir, planta, coletor, descricao_coletor)
        SELECT
          cadastro,
          data AS data,
          hora,
          dir,
          planta,
          coletor,
          descricao_coletor
        FROM evento_acesso_flat
        WHERE
          data IS NOT NULL
          AND hora IS NOT NULL
          AND (
            (
              dir = 'E'
              AND coletor LIKE '%REF%'
              AND coletor NOT LIKE '%GRILL%'
              AND coletor NOT LIKE '%GRIL%'
            )
            OR
            (
              dir = 'S'
              AND coletor LIKE '%PORT%'
              AND hora >= '11:00:00' AND hora <= '14:00:00'
            )
          );

        CREATE INDEX IF NOT EXISTS ix_almoco_cad_data          ON agg_almoco (cadastro, data);
        CREATE INDEX IF NOT EXISTS ix_almoco_data_planta_cad   ON agg_almoco (data, planta, cadastro);
        CREATE INDEX IF NOT EXISTS ix_almoco_data_coletor      ON agg_almoco (data, coletor);
    """)

    # =========================
    # CARGAS OPCIONAIS
    # =========================
    if isinstance(df_gptw, pd.DataFrame) and not df_gptw.empty:
        g = df_gptw.copy()
        g.columns = [c.strip() for c in g.columns]
        if "Área" in g.columns:
            g.rename(columns={"Área": "area"}, inplace=True)
        for y in ("2025", "2024", "2023"):
            if y in g.columns:
                g[y] = pd.to_numeric(g[y].replace({"**": np.nan}), errors="coerce")
        g2 = g[["area", "2025", "2024", "2023"]].rename(columns={"2025": "y2025", "2024": "y2024", "2023": "y2023"})
        g2 = g2.astype("object").where(g2.notna(), None)
        conn.execute("DELETE FROM gptw;")
        conn.executemany("INSERT INTO gptw(area,y2025,y2024,y2023) VALUES (?,?,?,?);", g2.itertuples(index=False, name=None))

    if isinstance(df_movimentacoes, pd.DataFrame) and not df_movimentacoes.empty:
        m = df_movimentacoes.copy()
        m.columns = [c.strip() for c in m.columns]
        rename = {
            "Cadastro": "cadastro",
            "Abreviação UO antiga": "abreviacao_uo_antiga",
            "Desc. UO antiga": "desc_uo_antiga",
            "Desc. Cargo antigo": "desc_cargo_antigo",
            "Abreviação UO": "abreviacao_uo",
            "Desc. UO": "desc_uo",
            "Desc. Cargo": "desc_cargo",
            "Tipo da Movimentação": "tipo_movimentacao",
            "Data de alteração": "data_alteracao",
        }
        m.rename(columns={k: v for k, v in rename.items() if k in m.columns}, inplace=True)
        if "data_alteracao" in m.columns:
            m["data_alteracao"] = pd.to_datetime(m["data_alteracao"], errors="coerce").dt.strftime("%Y-%m-%d")

        cols = ["cadastro","abreviacao_uo_antiga","desc_uo_antiga","desc_cargo_antigo","abreviacao_uo","desc_uo","desc_cargo","tipo_movimentacao","data_alteracao"]
        for c in cols:
            if c not in m.columns:
                m[c] = pd.NA
        m2 = m[cols].astype("object").where(m[cols].notna(), None)

        conn.execute("DELETE FROM movimentacoes;")
        conn.executemany(
            """INSERT INTO movimentacoes(
                cadastro,abreviacao_uo_antiga,desc_uo_antiga,desc_cargo_antigo,
                abreviacao_uo,desc_uo,desc_cargo,tipo_movimentacao,data_alteracao
            ) VALUES (?,?,?,?,?,?,?,?,?);""",
            m2.itertuples(index=False, name=None),
        )
        conn.executescript("""
            CREATE INDEX IF NOT EXISTS ix_mov_cadastro ON movimentacoes(cadastro);
            CREATE INDEX IF NOT EXISTS ix_mov_data ON movimentacoes(data_alteracao);
        """)

    # =========================
    # HEADCOUNT (FEITO NO ETL)
    # =========================
    if isinstance(df_headcount, pd.DataFrame) and not df_headcount.empty:
        h = df_headcount.copy()
        h.columns = [c.strip() for c in h.columns]
        if "Área" in h.columns:
            h.rename(columns={"Área": "area"}, inplace=True)
        for y in ("2023", "2024", "2025"):
            if y in h.columns:
                h[y] = pd.to_numeric(h[y], errors="coerce")
        h2 = h[["area", "2023", "2024", "2025"]].rename(columns={"2023": "y2023", "2024": "y2024", "2025": "y2025"})
        h2 = h2.astype("object").where(h2.notna(), None)
        conn.execute("DELETE FROM headcount;")
        conn.executemany("INSERT INTO headcount(area,y2023,y2024,y2025) VALUES (?,?,?,?);", h2.itertuples(index=False, name=None))
    else:
        # Se houver bases de RH disponíveis, calcula HC por ano (31/12) com roll-up por Heranca (área + subáreas)
        df_ativos_hr = globals().get("df_ativos", None)
        df_desligados_hr = globals().get("df_desligados", None)

        if isinstance(df_ativos_hr, pd.DataFrame) and isinstance(df_desligados_hr, pd.DataFrame) and (
            (not df_ativos_hr.empty) or (not df_desligados_hr.empty)
        ):
            try:
                hc_hr = compute_df_headcount_from_hr(
                    df_ativos=df_ativos_hr,
                    df_desligados=df_desligados_hr,
                    df_movimentacoes=df_movimentacoes if isinstance(df_movimentacoes, pd.DataFrame) else None,
                    anos=(2023, 2024, 2025),
                )
                hc_hr = hc_hr.astype("object").where(hc_hr.notna(), None)
                conn.execute("DELETE FROM headcount;")
                conn.executemany(
                    "INSERT INTO headcount(area,y2023,y2024,y2025) VALUES (?,?,?,?);",
                    hc_hr.itertuples(index=False, name=None),
                )
            except Exception as e:
                print("⚠️ Headcount via RH falhou; usando fallback por presença:", e)
                conn.executescript("""
            DELETE FROM headcount;

            DROP TABLE IF EXISTS tmp_cad_ano;
            CREATE TEMP TABLE tmp_cad_ano AS
            SELECT DISTINCT cadastro, ano
            FROM agg_registros_semanais;

            CREATE INDEX IF NOT EXISTS ix_tmp_cad_ano ON tmp_cad_ano (ano, cadastro);

            INSERT INTO headcount(area, y2023, y2024, y2025)
            WITH base AS (
              SELECT
                COALESCE(NULLIF(TRIM(c.area), ''), '(Sem Área)') AS area,
                t.ano AS ano,
                t.cadastro AS cadastro
              FROM tmp_cad_ano t
              JOIN dim_colaborador c
                ON c.cadastro = t.cadastro
            )
            SELECT
              area,
              COUNT(DISTINCT CASE WHEN ano = 2023 THEN cadastro END) AS y2023,
              COUNT(DISTINCT CASE WHEN ano = 2024 THEN cadastro END) AS y2024,
              COUNT(DISTINCT CASE WHEN ano = 2025 THEN cadastro END) AS y2025
            FROM base
            GROUP BY area
            ORDER BY area;

            DROP TABLE IF EXISTS tmp_cad_ano;
        """)
        else:
            # Fallback: proxy via presença (agg_registros_semanais + dim_colaborador)
            conn.executescript("""
            DELETE FROM headcount;

            DROP TABLE IF EXISTS tmp_cad_ano;
            CREATE TEMP TABLE tmp_cad_ano AS
            SELECT DISTINCT cadastro, ano
            FROM agg_registros_semanais;

            CREATE INDEX IF NOT EXISTS ix_tmp_cad_ano ON tmp_cad_ano (ano, cadastro);

            INSERT INTO headcount(area, y2023, y2024, y2025)
            WITH base AS (
              SELECT
                COALESCE(NULLIF(TRIM(c.area), ''), '(Sem Área)') AS area,
                t.ano AS ano,
                t.cadastro AS cadastro
              FROM tmp_cad_ano t
              JOIN dim_colaborador c
                ON c.cadastro = t.cadastro
            )
            SELECT
              area,
              COUNT(DISTINCT CASE WHEN ano = 2023 THEN cadastro END) AS y2023,
              COUNT(DISTINCT CASE WHEN ano = 2024 THEN cadastro END) AS y2024,
              COUNT(DISTINCT CASE WHEN ano = 2025 THEN cadastro END) AS y2025
            FROM base
            GROUP BY area
            ORDER BY area;

            DROP TABLE IF EXISTS tmp_cad_ano;
        """)

    # Limpeza + estatísticas
    conn.executescript("""
        DROP TABLE IF EXISTS tmp_presenca_validos;
        DROP TABLE IF EXISTS tmp_last;
        DROP TABLE IF EXISTS tmp_last_rid;
    """)
    conn.execute("ANALYZE;")
    try:
        conn.execute("PRAGMA optimize;")
    except Exception:
        pass

    # Finaliza banco para leitura (sem WAL)
    try:
        conn.execute("PRAGMA journal_mode = DELETE;")
        conn.execute("PRAGMA synchronous = NORMAL;")
    except Exception:
        pass
    conn.execute("PRAGMA locking_mode = NORMAL;")

finally:
    conn.close()

print("✅ DW gerado em:", SQLITE_PATH)


In [ ]:
'''# Importando bibliotecas para a criação desse relatório
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# =========================================================
# 1) LOAD + FILTRO
# =========================================================

# Garantir tipos ANTES do filtro
df_ronda_filtrado["data"] = pd.to_datetime(df_ronda_filtrado["data"], errors="coerce").dt.normalize()
df_ronda_filtrado["planta"] = df_ronda_filtrado["planta"].astype(str)

# Se ainda não existir, cria "area" pra não quebrar
if "area" not in df_ronda_filtrado.columns:
    df_ronda_filtrado["area"] = "Sem Área"
df_ronda_filtrado["area"] = df_ronda_filtrado["area"].fillna("Sem Área")

# Semanas analisadas
semanas = [
    ("2025-04-28", "2025-05-04"),  # 01/05 (qui) — Dia do Trabalho
    ("2025-06-16", "2025-06-22"),  # 19/06 (qui) — Corpus Christi
    ("2025-11-17", "2025-11-23"),  # 20/11 (qui) — Consciência Negra
    ("2025-12-22", "2025-12-28"),  # 25/12 (qui) — Natal
    ("2026-02-16", "2026-02-22"),  # 17/02 (ter) — Carnaval
    ("2026-04-20", "2026-04-26"),  # 21/04 (ter) — Tiradentes
    ("2026-06-01", "2026-06-07"),  # 04/06 (qui) — Corpus Christi
]

# Filtro por herança + semanas
mask_periodo = pd.Series(False, index=df_ronda_filtrado.index)
for ini, fim in semanas:
    ini = pd.to_datetime(ini)
    fim = pd.to_datetime(fim)
    mask_periodo = mask_periodo | df_ronda_filtrado["data"].between(ini, fim)

df_ronda_filtrado = df_ronda_filtrado.loc[
    (df_ronda_filtrado["heranca"] == "AUDITORIA INTERNA") & mask_periodo
].copy()

# Tag de semana (início)
conds = [df_ronda_filtrado["data"].between(pd.to_datetime(ini), pd.to_datetime(fim)) for ini, fim in semanas]
week_starts = [pd.to_datetime(ini) for ini, _ in semanas]
df_ronda_filtrado["semana_inicio"] = np.select(conds, week_starts, default=pd.NaT)

# Presencial vs Home (planta 999 = home)
df_ronda_filtrado["presencial"] = df_ronda_filtrado["planta"] != "999"

# =========================================================
# 2) RESUMO
# =========================================================
gcols = ["cadastro", "nome", "area", "semana_inicio"]

dias_com_registro = df_ronda_filtrado.groupby(gcols)["data"].nunique().rename("dias_com_registro")

dias_presenciais = (
    df_ronda_filtrado[df_ronda_filtrado["presencial"]]
    .groupby(gcols)["data"].nunique().rename("dias_presenciais")
)

dias_home = (
    df_ronda_filtrado[~df_ronda_filtrado["presencial"]]
    .groupby(gcols)["data"].nunique().rename("dias_home")
)

resumo = (
    pd.concat([dias_com_registro, dias_presenciais, dias_home], axis=1)
      .fillna({"dias_com_registro": 0, "dias_presenciais": 0, "dias_home": 0})
      .reset_index()
)

resumo[["dias_com_registro", "dias_presenciais", "dias_home"]] = resumo[
    ["dias_com_registro", "dias_presenciais", "dias_home"]
].astype(int)

resumo["semana_fim"] = resumo["semana_inicio"] + pd.Timedelta(days=6)

# Quem NÃO cumpriu 3 dias presenciais (IGNORANDO quem teve 0 presencial)
nao_cumpriu = (
    resumo[resumo["dias_presenciais"] < 3]
      .sort_values(["semana_inicio", "dias_presenciais", "cadastro"])
      .reset_index(drop=True)
)

# =========================================================
# >>> ALTERAÇÃO PRINCIPAL: NÃO contar/plotar dias_presenciais == 0 nos VISUAIS
# =========================================================

# Flag consistente com a regra do nao_cumpriu (zero NÃO entra como não conformidade)
resumo["nao_cumpriu_3"] = (resumo["dias_presenciais"] < 3)

# semana stats (gráfico D) — calculado na base sem zeros
semana_stats = (
    resumo.groupby("semana_inicio", as_index=False)
       .agg(
           pessoa_semanas=("cadastro", "size"),
           pessoas_unicas=("cadastro", "nunique"),
           pct_nao_cumpriu_3=("nao_cumpriu_3", "mean"),
           media_dias_presenciais=("dias_presenciais", "mean"),
       )
)

# =========================================================
# 3) DASHBOARD (8 GRÁFICOS)
# =========================================================

# Paleta “Volks-like”
VW_NAVY   = "#001E50"
VW_BLUE   = "#0B5FA5"
VW_BLUE2  = "#7FB0D6"
VW_GREY   = "#C9D1D9"
VW_DARK   = "#1B2A41"
ACCENT    = "#FFB000"
BG_LIGHT  = "#F6F8FB"

plt.rcParams.update({
    "figure.facecolor": BG_LIGHT,
    "axes.facecolor": "white",
    "axes.edgecolor": VW_GREY,
    "axes.labelcolor": VW_DARK,
    "xtick.color": VW_DARK,
    "ytick.color": VW_DARK,
    "text.color": VW_DARK,
    "font.size": 10,
})

VW_CMAP = LinearSegmentedColormap.from_list(
    "vw_blues",
    [BG_LIGHT, "#CFE2F3", VW_BLUE, VW_NAVY]
)

def add_desc(ax, text):
    ax.text(
        0.01, -0.35, text,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=9, color="#334155",
        wrap=True,
        bbox=dict(boxstyle="round,pad=0.45", facecolor=BG_LIGHT, edgecolor=VW_GREY, linewidth=0.9)
    )

def style_card(ax):
    for s in ax.spines.values():
        s.set_visible(True)
        s.set_color(VW_GREY)
        s.set_linewidth(1.2)
    ax.set_facecolor("white")

def make_first_name_labels(series_names: pd.Series) -> pd.Series:
    first = series_names.astype(str).str.strip().str.split().str[0].replace({"": "Semnome"})
    counts = {}
    labels = []
    for fn in first.tolist():
        counts[fn] = counts.get(fn, 0) + 1
        labels.append(fn if counts[fn] == 1 else f"{fn} ({counts[fn]})")
    return pd.Series(labels, index=series_names.index)


# =========================================================
# DESCRIÇÕES (mais interpretáveis / focadas no "o que significa")
# =========================================================
DESC = {
    "A": (
        "Resumo:\n"
        "• Existe uma área que puxa o problema: ela aparece com a maior taxa de semanas <3 dias.\n"
        "• Como os volumes entre áreas são parecidos, a diferença real é o comportamento (taxa), não o tamanho.\n"
        "O que isso sugere:\n"
        "• Prioridade é atuar nessa área líder (é o maior “risco” do painel).\n"
        "Próximo passo:\n"
        "• Quebrar essa área por gestor/time/turno para encontrar o foco exato."
    ),
    "B": (
        "Resumo:\n"
        "• O volume total de não conformidade está concentrado em 1 área (as outras contribuem bem menos).\n"
        "O que isso sugere:\n"
        "• Se você atacar a 1ª área do ranking, você reduz a maior parte do problema mais rápido.\n"
        "Próximo passo:\n"
        "• Tratar como plano 80/20: ação imediata na líder, monitorar as demais."
    ),
    "C": (
        "Resumo:\n"
        "• Uma área mostra instabilidade (varia bastante entre semanas e chega a ter pico muito alto).\n"
        "• Outras áreas têm problema pontual (aparece em uma semana e some nas demais).\n"
        "O que isso sugere:\n"
        "• A principal área tem padrão recorrente; as demais parecem efeito de evento específico.\n"
        "Próximo passo:\n"
        "• Na semana do pico: cruzar com agenda, férias, viagens, reuniões externas e comunicação interna."
    ),
    "D": (
        "Resumo:\n"
        "• Quando a média de dias presenciais cai, o % de não conformidade tende a subir (e vice-versa).\n"
        "• Existe uma semana claramente pior que as outras (pico de não conformidade).\n"
        "O que isso sugere:\n"
        "• O problema é sensível a contexto/rotina da semana (evento, agenda, exceções).\n"
        "Próximo passo:\n"
        "• Usar essa semana como “caso” para investigação (o que mudou nela?)."
    ),
    "E": (
        "Resumo:\n"
        "• A maioria está no alvo (3 dias), e a segunda maior fatia fica muito perto (2 dias).\n"
        "• Casos extremos (1 dia) são poucos.\n"
        "O que isso sugere:\n"
        "• O ganho mais rápido é converter quem está em 2 dias para 3 dias (ajuste fino).\n"
        "Próximo passo:\n"
        "• Direcionar comunicação/combinação de agenda para o grupo de 2 dias."
    ),
    "F": (
        "Resumo:\n"
        "• Aparecem poucos Nomes com recorrência (repetem <3 dias em mais de uma semana).\n"
        "• O restante tende a ser casos pontuais (uma ocorrência).\n"
        "O que isso sugere:\n"
        "• Você tem “perfil recorrente” (merece checagem de exceção) e “perfil pontual” (evento).\n"
        "Próximo passo:\n"
        "• Recorrentes: validar justificativa/arranjo formal; Pontuais: cruzar com a semana do pico."
    ),
    "G": (
        "Resumo:\n"
        "• Mostra quem tem maior presença acumulada no período (referência de comportamento).\n"
        "O que isso sugere:\n"
        "• Serve como benchmark de rotina/agenda que funciona.\n"
        "Próximo passo:\n"
        "• Comparar a rotina dos mais presentes com a área crítica (o que eles fazem diferente?)."
    ),
    "H": (
        "Resumo:\n"
        "• Mostra quem é consistente (cumpre em várias semanas, não só “média alta”).\n"
        "O que isso sugere:\n"
        "• Melhor indicador de regularidade do que o total (G).\n"
        "Próximo passo:\n"
        "• Replicar boas práticas dos consistentes na área crítica (agenda, rituais, acordos de presença)."
    ),
}


# Base estatística pro dashboard (já sem zeros via resumo)
area_agg = (
    resumo.groupby("area", dropna=False, as_index=False)
       .agg(
           pessoa_semanas=("semana_inicio", "size"),
           pessoas_unicas=("cadastro", "nunique"),
           nao_cumpriu_count=("nao_cumpriu_3", "sum"),
           pct_nao_cumpriu=("nao_cumpriu_3", "mean"),
           media_dias_presenciais=("dias_presenciais", "mean"),
       )
)
area_agg["score_risco"] = area_agg["pct_nao_cumpriu"] * np.sqrt(area_agg["pessoa_semanas"])

MIN_PESSOA_SEMANAS = 10
area_base = area_agg[area_agg["pessoa_semanas"] >= MIN_PESSOA_SEMANAS].copy()
area_top = (
    area_base.sort_values(["score_risco","pct_nao_cumpriu","pessoa_semanas"], ascending=[False, False, False])
             .head(10)
             .copy()
)

pessoa_stats = (
    resumo.groupby(["cadastro","nome","area"], as_index=False)
       .agg(
           semanas_total=("semana_inicio","nunique"),
           semanas_nao_cumpriu=("nao_cumpriu_3","sum"),
           dias_home_total=("dias_home","sum"),
           dias_presenciais_total=("dias_presenciais","sum"),
       )
)
pessoa_stats["taxa_nao_cumpriu"] = pessoa_stats["semanas_nao_cumpriu"] / pessoa_stats["semanas_total"] * 100
pessoa_stats["semanas_cumpriu"] = pessoa_stats["semanas_total"] - pessoa_stats["semanas_nao_cumpriu"]
pessoa_stats["taxa_cumpriu"] = pessoa_stats["semanas_cumpriu"] / pessoa_stats["semanas_total"] * 100

# Painel 4x2
fig = plt.figure(figsize=(18, 26))
gs = fig.add_gridspec(4, 2, hspace=0.85, wspace=0.28)

# ---------------------------------------------------------
# (A) Prioridade — Taxa x Volume
# ---------------------------------------------------------
axA = fig.add_subplot(gs[0, 0]); style_card(axA)

dfA = area_base.copy()
dfA["pct"] = dfA["pct_nao_cumpriu"] * 100
x_med = dfA["pessoa_semanas"].median()
y_med = dfA["pct"].median()
topN = dfA.sort_values("score_risco", ascending=False).head(8)

axA.scatter(dfA["pessoa_semanas"], dfA["pct"],
            s=(dfA["pessoas_unicas"] * 10).clip(40, 900),
            alpha=0.65, color=VW_BLUE2, edgecolors=VW_NAVY, linewidths=0.6)
axA.scatter(topN["pessoa_semanas"], topN["pct"],
            s=(topN["pessoas_unicas"] * 12).clip(70, 1100),
            alpha=0.95, color=ACCENT, edgecolors=VW_NAVY, linewidths=0.8)

for _, r in topN.iterrows():
    axA.annotate(str(r["area"]),
                 xy=(r["pessoa_semanas"], r["pct"]),
                 xytext=(18, 14),
                 textcoords="offset points",
                 ha="left", va="bottom", fontsize=8,
                 bbox=dict(boxstyle="round,pad=0.18", facecolor="white", edgecolor=VW_GREY, linewidth=0.6, alpha=0.9))

axA.axvline(x_med, linestyle="--", color=VW_GREY, linewidth=1)
axA.axhline(y_med, linestyle="--", color=VW_GREY, linewidth=1)
axA.set_title("A) Prioridade — Taxa × Volume por Área", pad=10, color=VW_NAVY, fontweight="bold")
axA.set_xlabel("Volume (pessoa-semanas)")
axA.set_ylabel("% com < 3 dias presenciais")
axA.grid(True, linestyle="--", alpha=0.18)

axA.legend(handles=[
    Line2D([0],[0], marker='o', color='w', label='Áreas (geral)',
           markerfacecolor=VW_BLUE2, markeredgecolor=VW_NAVY, markersize=9),
    Line2D([0],[0], marker='o', color='w', label='Top 8 (maior risco ajustado por volume)',
           markerfacecolor=ACCENT, markeredgecolor=VW_NAVY, markersize=9),
], loc="upper right", frameon=True, facecolor="white", edgecolor=VW_GREY)

add_desc(axA, DESC["A"])

# ---------------------------------------------------------
# (B) Pareto — SOMENTE BARRAS
# ---------------------------------------------------------
axB = fig.add_subplot(gs[0, 1]); style_card(axB)

dfB = area_base.sort_values("nao_cumpriu_count", ascending=False).head(12).copy()
dfB["taxa"] = (dfB["pct_nao_cumpriu"] * 100).round(1)

axB.bar(dfB["area"].astype(str), dfB["nao_cumpriu_count"], color=VW_NAVY, alpha=0.92)
axB.set_title("B) Pareto — Massa de não conformidade (volume absoluto)", pad=10, color=VW_NAVY, fontweight="bold")
axB.set_xlabel("Área")
axB.set_ylabel("Qtd de pessoa-semanas < 3 dias")
axB.grid(axis="y", linestyle="--", alpha=0.18)
axB.tick_params(axis="x", rotation=35)

max_bar = float(dfB["nao_cumpriu_count"].max())
axB.set_ylim(0, max_bar * 1.35)

for i, r in enumerate(dfB.itertuples(index=False)):
    axB.text(i, r.nao_cumpriu_count + max_bar*0.03, f"{int(r.nao_cumpriu_count)}\n({r.taxa}%)",
             ha="center", va="bottom", fontsize=8, color=VW_DARK)

legendB = [Patch(facecolor=VW_NAVY, edgecolor=VW_NAVY, label="Volume: pessoa-semanas < 3 dias (barra)")]
axB.legend(handles=legendB, loc="upper right", frameon=True, facecolor="white", edgecolor=VW_GREY)

add_desc(axB, DESC["B"])

# ---------------------------------------------------------
# (C) Heatmap — % não cumpriu por Área × semana (Top 10)
# ---------------------------------------------------------
axC = fig.add_subplot(gs[1, 0]); style_card(axC)

aw = (resumo[resumo["area"].astype(str).isin(area_top["area"].astype(str))]
      .groupby(["area","semana_inicio"], as_index=False)["nao_cumpriu_3"]
      .mean()
      .rename(columns={"nao_cumpriu_3": "pct"}))
aw["pct"] *= 100

pivot = aw.pivot_table(index="area", columns="semana_inicio", values="pct", aggfunc="mean")
pivot = pivot.reindex(area_top["area"].astype(str).tolist())
pivot = pivot.reindex(sorted(pivot.columns), axis=1)

mat = pivot.values
im = axC.imshow(mat, aspect="auto", cmap=VW_CMAP, vmin=0, vmax=100)
axC.set_title("C) Estabilidade — % não cumpriu por Área × semana (Top 10)", pad=10, color=VW_NAVY, fontweight="bold")
axC.set_xlabel("semana (início)")
axC.set_ylabel("Área")

xlabels = [pd.to_datetime(c).strftime("%Y-%m-%d") for c in pivot.columns]
axC.set_xticks(np.arange(len(xlabels)))
axC.set_xticklabels(xlabels, rotation=45, ha="right")
axC.set_yticks(np.arange(pivot.shape[0]))
axC.set_yticklabels(pivot.index.astype(str))

for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat[i, j]
        if np.isfinite(v):
            axC.text(j, i, f"{v:.0f}%", ha="center", va="center",
                     fontsize=8, color=("white" if v >= 55 else VW_DARK))

cbar = fig.colorbar(im, ax=axC, fraction=0.046, pad=0.04)
cbar.set_label("% não cumprimento (<3 dias)")

add_desc(axC, DESC["C"])

# ---------------------------------------------------------
# (D) Tendência semanal — % não cumpriu vs média presencial
# ---------------------------------------------------------
axD = fig.add_subplot(gs[1, 1]); style_card(axD)

dfD = semana_stats.sort_values("semana_inicio").copy()
dfD["semana"] = dfD["semana_inicio"].dt.strftime("%Y-%m-%d")

pct = (dfD["pct_nao_cumpriu_3"] * 100).to_numpy()
media = dfD["media_dias_presenciais"].to_numpy()

l1, = axD.plot(dfD["semana"], pct, marker="o", color=VW_NAVY, linewidth=2.4, label="% não cumpriu")
axD.set_title("D) Evolução semanal — % não cumpriu vs média presencial", pad=10, color=VW_NAVY, fontweight="bold")
axD.set_xlabel("semana (início)")
axD.set_ylabel("% não cumpriu")
axD.grid(True, linestyle="--", alpha=0.18)
axD.tick_params(axis="x", rotation=45)

axD2 = axD.twinx()
l2, = axD2.plot(dfD["semana"], media, marker="o", color=VW_BLUE, linewidth=2.2, label="Média dias presenciais")
axD2.set_ylabel("Média dias presenciais")

axD.legend(handles=[l1, l2], loc="upper left", frameon=True, facecolor="white", edgecolor=VW_GREY)

for x, y in zip(dfD["semana"], pct):
    axD.text(x, y + 0.6, f"{y:.1f}%", ha="center", fontsize=8)

add_desc(axD, DESC["D"])

# ---------------------------------------------------------
# (E) Severidade — distribuição de dias presenciais (sem 0)
# ---------------------------------------------------------
axE = fig.add_subplot(gs[2, 0]); style_card(axE)

dist = (resumo.groupby("dias_presenciais").size().rename("qtd").reset_index().sort_values("dias_presenciais"))
dist["pct"] = dist["qtd"] / dist["qtd"].sum() * 100

axE.bar(dist["dias_presenciais"].astype(int).astype(str), dist["pct"], color=VW_BLUE, alpha=0.92)
axE.set_title("E) Severidade — Distribuição de dias presenciais", pad=10, color=VW_NAVY, fontweight="bold")
axE.set_xlabel("Dias presenciais na semana")
axE.set_ylabel("% de pessoa-semanas")
axE.grid(axis="y", linestyle="--", alpha=0.18)

ymaxE = float(dist["pct"].max()) * 1.25
axE.set_ylim(0, ymaxE)
for i, r in dist.iterrows():
    axE.text(i, r["pct"] + ymaxE*0.02, f"{r['pct']:.1f}%", ha="center", fontsize=8)

add_desc(axE, DESC["E"])

# ---------------------------------------------------------
# (F) Investigação — recorrência (<3 dias) por pessoa
# ---------------------------------------------------------
axF = fig.add_subplot(gs[2, 1]); style_card(axF)

topF = (pessoa_stats.sort_values(["semanas_nao_cumpriu","taxa_nao_cumpriu","dias_home_total"],
                                ascending=[False, False, False])
                    .head(12)
                    .copy())
topF["primeiro_nome"] = make_first_name_labels(topF["nome"])

valsF = topF["semanas_nao_cumpriu"].to_numpy()
axF.bar(topF["primeiro_nome"], valsF, color=VW_NAVY, alpha=0.92)
axF.set_title("F) Investigação — Recorrência (<3 dias) por pessoa", pad=10, color=VW_NAVY, fontweight="bold")
axF.set_xlabel("Primeiro nome")
axF.set_ylabel("Qtd de semanas < 3 dias")
axF.grid(axis="y", linestyle="--", alpha=0.18)

axF.tick_params(axis="x", rotation=45)
for lab in axF.get_xticklabels():
    lab.set_ha("right")

axF.set_ylim(0, max(valsF.max()*1.35, valsF.max()+1))
for i, r in enumerate(topF.itertuples(index=False)):
    axF.text(i, r.semanas_nao_cumpriu + 0.08, f"{str(r.area)}\n{r.taxa_nao_cumpriu:.0f}%",
             ha="center", va="bottom", fontsize=8)

add_desc(axF, DESC["F"])

# ---------------------------------------------------------
# (G) Quem mais vem — dias presenciais total
# ---------------------------------------------------------
axG = fig.add_subplot(gs[3, 0]); style_card(axG)

topG = (pessoa_stats.sort_values(["dias_presenciais_total","semanas_cumpriu","taxa_cumpriu"],
                                ascending=[False, False, False])
                    .head(12)
                    .copy())
topG["primeiro_nome"] = make_first_name_labels(topG["nome"])

valsG = topG["dias_presenciais_total"].to_numpy()
axG.bar(topG["primeiro_nome"], valsG, color=VW_BLUE, alpha=0.92)
axG.set_title("G) Quem mais vem — Dias presenciais (total)", pad=10, color=VW_NAVY, fontweight="bold")
axG.set_xlabel("Primeiro nome")
axG.set_ylabel("Dias presenciais (soma)")
axG.grid(axis="y", linestyle="--", alpha=0.18)

axG.tick_params(axis="x", rotation=45)
for lab in axG.get_xticklabels():
    lab.set_ha("right")

axG.set_ylim(0, max(valsG.max()*1.25, valsG.max()+1))
for i, r in enumerate(topG.itertuples(index=False)):
    axG.text(i, r.dias_presenciais_total + 0.2, f"{str(r.area)}\n{r.taxa_cumpriu:.0f}%",
             ha="center", va="bottom", fontsize=8)

add_desc(axG, DESC["G"])

# ---------------------------------------------------------
# (H) Quem mais vem — consistência (semanas cumprindo)
# ---------------------------------------------------------
axH = fig.add_subplot(gs[3, 1]); style_card(axH)

baseH = pessoa_stats[pessoa_stats["semanas_total"] >= 2].copy()
topH = (baseH.sort_values(["semanas_cumpriu","taxa_cumpriu","dias_presenciais_total"],
                         ascending=[False, False, False])
             .head(12)
             .copy())
topH["primeiro_nome"] = make_first_name_labels(topH["nome"])

valsH = topH["semanas_cumpriu"].to_numpy()
axH.bar(topH["primeiro_nome"], valsH, color=VW_NAVY, alpha=0.85)
axH.set_title("H) Quem mais vem — Consistência (semanas ≥3 dias)", pad=10, color=VW_NAVY, fontweight="bold")
axH.set_xlabel("Primeiro nome")
axH.set_ylabel("Qtd semanas cumprindo")
axH.grid(axis="y", linestyle="--", alpha=0.18)

axH.tick_params(axis="x", rotation=45)
for lab in axH.get_xticklabels():
    lab.set_ha("right")

axH.set_ylim(0, max(valsH.max()*1.25, valsH.max()+1))
for i, r in enumerate(topH.itertuples(index=False)):
    axH.text(i, r.semanas_cumpriu + 0.08, f"{str(r.area)}\n{r.taxa_cumpriu:.0f}%",
             ha="center", va="bottom", fontsize=8)

add_desc(axH, DESC["H"])

fig.suptitle(
    "Painel de Compliance dos dias Presenciais — Auditoria Interna (8 insights)",
    fontsize=20, fontweight="bold", color=VW_NAVY, y=0.92
)

# =========================================================
# 4) EXPORT PNG + PDF + SAÍDAS
# =========================================================
OUT_PDF = "Relatório da Auditoria Interna.pdf"

# PDF com o painel (mesma figura)
fig.savefig(OUT_PDF, bbox_inches="tight")

plt.show()

print((nao_cumpriu.groupby('nome').size().reset_index(name='qtd_aparicoes')).to_string(index=False))

#                              nome  qtd_aparicoes
#          BEATRIZ BARBOSA DOS REIS              2
#    BRUNO SILVA MANGUEIRA DE ASSIS              1
#   CARLOS RAIMUNDO DE JESUS JUNIOR              1
#           DARIO ALMEIDA DE MACEDO              1
#FILLIPE FONTENELE DE QUEIROZ TAHIM              1
#            NANCY SALVADORI BITTAR              2
#NATHALIA DA COSTA HADDAD RODRIGUES              1
#            RAFAEL MAXIMO DA SILVA              1
#      REGINA KEIKO YOSHITAKE ROCHA              1
#                    TATHIANA BUDRI              1

print(nao_cumpriu.to_string(index=False))

#cadastro                               nome area semana_inicio  dias_com_registro  dias_presenciais  dias_home  semana_fim
#4028435             NANCY SALVADORI BITTAR B-IA   2025-04-28                  1                 1          0 2025-05-04
#4002192       REGINA KEIKO YOSHITAKE ROCHA B-IA   2025-04-28                  3                 2          1 2025-05-04
#4028430     BRUNO SILVA MANGUEIRA DE ASSIS B-IT   2025-04-28                  3                 2          1 2025-05-04
#4028436                     TATHIANA BUDRI B-IT   2025-04-28                  3                 2          1 2025-05-04
#4063741 FILLIPE FONTENELE DE QUEIROZ TAHIM B-IE   2025-06-16                  4                 0          4 2025-06-22
#4065899           BEATRIZ BARBOSA DOS REIS B-IA   2025-06-16                  3                 2          1 2025-06-22
#4028435             NANCY SALVADORI BITTAR B-IA   2025-11-17                  2                 2          0 2025-11-23
#4044662            DARIO ALMEIDA DE MACEDO B-IE   2025-11-17                  3                 2          1 2025-11-23
#4052158 NATHALIA DA COSTA HADDAD RODRIGUES B-IA   2025-11-17                  3                 2          1 2025-11-23
#4053306             RAFAEL MAXIMO DA SILVA B-IE   2025-11-17                  3                 2          1 2025-11-23
#4061194    CARLOS RAIMUNDO DE JESUS JUNIOR B-IA   2025-11-17                  3                 2          1 2025-11-23
#4065899           BEATRIZ BARBOSA DOS REIS B-IA   2025-11-17                  3                 2          1 2025-11-23

'''